In [11]:
import gc
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split

# ==========================================
# CONFIGURATIE
# ==========================================
DATA_PATH = "transformed_data_sampled.csv"
TARGET_COL = "status"  # De doelvariabele die nu 0, 1, 2... bevat
IS_CLASSIFICATION = True  # Moet True zijn voor statuscode-voorspelling

# ==========================================
# 1. GEHEUGENEFFICIËNT DATA INLADEN (DOWNCASTING)
# ==========================================
print("1. Dtypes analyseren voor geheugenbesparing...")
sample = pd.read_csv(DATA_PATH, nrows=100)

dtypes = {}
for col in sample.columns:
    if sample[col].dtype == "float64":
        dtypes[col] = "float32"
    elif sample[col].dtype == "int64":
        dtypes[col] = "int32"
    else:
        dtypes[col] = sample[col].dtype

print("-> Data inladen met geoptimaliseerde dtypes...")
df = pd.read_csv(DATA_PATH, dtype=dtypes)

# Splits direct in X en y
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

del df
gc.collect()

# ==========================================
# 2. TRAIN / VALIDATION SPLIT
# ==========================================
print("2. Dataset opsplitsen...")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === DYNAMISCH AANTAL KLASSEN BEPALEN ===
# Omdat de data al gecodeerd is als 0, 1, 2... kunnen we unieke waarden tellen
num_classes = int(y_train.nunique())
print(f"-> Aantal gedetecteerde klassen (statuscodes) voor training: {num_classes}")

del X, y
gc.collect()

# ==========================================
# 3. OMZETTEN NAAR NATIVE XGBOOST DMATRIX
# ==========================================
print("3. DMatrix aanmaken (interne XGBoost structuur)...")
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

del X_train, X_val, y_train, y_val
gc.collect()

# ==========================================
# 4. PARAMETERS OPTIMALISEREN VOOR MULTICLASS & 8GB RAM
# ==========================================
# 'multi:softprob' geeft de kansen per statuscode (bijv. [0.1, 0.8, 0.1] voor 3 klassen)
objective = "multi:softprob" if IS_CLASSIFICATION else "reg:squarederror"
eval_metric = "mlogloss" if IS_CLASSIFICATION else "rmse"  # 'mlogloss' is vereist voor multiclass

params = {
    "objective": objective,
    "num_class": num_classes,
    "tree_method": "hist",
    "max_depth": 6,          
    "learning_rate": 0.05,
    "eval_metric": eval_metric,
    
    # --- VOEG DEZE TOE VOOR EXTRA PERFORMANCE ---
    "subsample": 0.8,        # Traint elke boom op 80% van de rijen (voorkomt overfitting)
    "colsample_bytree": 0.8, # Gebruikt 80% van de kolommen per boom (dwingt variatie af)
    
    "verbosity": 1,
}

# ==========================================
# 5. MODEL TRAINEN
# ==========================================
print("4. Starten met trainen...")
evallist = [(dval, "validation"), (dtrain, "train")]
num_round = 2000  # Hoog aantal, we stoppen toch vroeg via early stopping

bst = xgb.train(
    params,
    dtrain,
    num_boost_round=num_round,
    evals=evallist,
    early_stopping_rounds=15,  # Stopt als het model 15 bomen lang niet verbetert
    verbose_eval=50,  # Print status elke 50 bomen
)

print("\nTraining succesvol afgerond!")

# Optioneel: Model opslaan
# bst.save_model("xgboost_multiclass_statuscodes.json")

1. Dtypes analyseren voor geheugenbesparing...
-> Data inladen met geoptimaliseerde dtypes...
2. Dataset opsplitsen...
-> Aantal gedetecteerde klassen (statuscodes) voor training: 7
3. DMatrix aanmaken (interne XGBoost structuur)...
4. Starten met trainen...
[0]	validation-mlogloss:0.94512	train-mlogloss:0.94804
[50]	validation-mlogloss:0.73569	train-mlogloss:0.73205
[100]	validation-mlogloss:0.71625	train-mlogloss:0.70971
[150]	validation-mlogloss:0.70805	train-mlogloss:0.69924
[200]	validation-mlogloss:0.70239	train-mlogloss:0.69152
[250]	validation-mlogloss:0.69801	train-mlogloss:0.68528
[300]	validation-mlogloss:0.69460	train-mlogloss:0.68016
[350]	validation-mlogloss:0.69153	train-mlogloss:0.67539
[400]	validation-mlogloss:0.68891	train-mlogloss:0.67120
[450]	validation-mlogloss:0.68697	train-mlogloss:0.66793
[500]	validation-mlogloss:0.68525	train-mlogloss:0.66490
[550]	validation-mlogloss:0.68412	train-mlogloss:0.66259
[600]	validation-mlogloss:0.68313	train-mlogloss:0.66046
[65

In [10]:
# Train een mini-model met 5 bomen om de belangrijkste feature te vinden
mini_bst = xgb.train(params, dtrain, num_boost_round=5)
importance = mini_bst.get_score(importance_type="gain")
print("Belangrijkste features:", sorted(importance.items(), key=lambda x: x[1], reverse=True)[:3])


Belangrijkste features: [('type_GET', 7014.96728515625), ('body_MISSING', 222.2132568359375), ('type_POST', 112.40474700927734)]
